In [40]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [41]:
load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b")

In [42]:
class BlogState(TypedDict):
    topic: str
    outline: str
    content: str
    blog_score: int

In [43]:
def generate_outline(state=BlogState):
    topic = state["topic"]
    prompt = f"Create an outline for a blog on topic {topic}"
    response = model.invoke(prompt)
    outline = response.content
    state["outline"] = outline
    return state

In [44]:
def generate_blog(state=BlogState):
    topic = state["topic"]
    outline = state["outline"]
    response = model.invoke([
        {"role": "system", "content": "Create a blog using the provided outline for the topic" },
        {"role": "user", "content": f"topic: {topic} outline: {outline}"}
    ])
    content = response.content
    state["content"] = content
    return state

In [45]:
def score_blog(state: BlogState):
    outline = state["outline"]
    content = state["content"]

    prompt = f"Based on the outline {outline}, score the blog content {content} between 1 to 10"
    score = (model.invoke(prompt)).content
    state["blog_score"] = score
    return state

In [46]:
graph = StateGraph(BlogState)

graph.add_node("generate_outline", generate_outline)
graph.add_node("generate_blog", generate_blog)
graph.add_node("score_blog", score_blog)

graph.add_edge(START, 'generate_outline')
graph.add_edge('generate_outline', 'generate_blog')
graph.add_edge('generate_blog', 'score_blog')
graph.add_edge('score_blog', END)

workflow = graph.compile()

initial_state = {"topic": "How manage ADHD as a software engineer"}
output = workflow.invoke(initial_state)
# print(output)

In [47]:
# print(output["content"])
print(output["blog_score"])

**Score: 9/10**

---

### Why the Blog Earns a 9

| Dimension | Strengths | Minor Weaknesses |
|-----------|-----------|------------------|
| **Relevance & Focus** | The post zeroes in on the exact pain‑points of ADHD‑affected software engineers—productivity, collaboration, burnout, career progression—and offers concrete, tech‑centric solutions. | None; the focus is spot‑on. |
| **Structure & Flow** | The article follows the provided outline perfectly, with a clear hook, logical progression, and a strong call‑to‑action. The use of tables (symptoms → coding manifestations; pain points → challenges) makes complex ideas digestible. | A few transitions could be smoother (e.g., moving from “How ADHD Plays Out in Code” to “The Pain Points”). |
| **Depth & Practicality** | Each section delivers actionable tactics: Pomodoro tweaks, time‑blocking, specific tools, workspace hacks, and even legal rights. The “Self‑Diagnosis Checklist” is a useful quick‑reference. | Some sections (e.g., “Career Gr